In [8]:
"""
NFStream PCAP → Windowed Multicast IDS Training Dataset
========================================================
Use Script 2 (Document 16) as the base — it has more features.
All corrections applied. See CORRECTIONS section below.

CORRECTIONS vs previous versions:
  1. igmp_join detection: now captures BOTH IGMPv2 (dst=group addr 239.x.x.x)
     AND IGMPv3 (dst=224.0.0.22). Previous version missed all benign IGMPv2 joins.
  2. label_window class 2: added unique_src_ips_igmp > 1 — spoof sends from
     victim IPs so multiple IGMP sources in one window is the real indicator.
  3. label_window class 6: added igmp_query_rate check alongside group_churn_rate.
  4. label_window class 1: added unique_mcast_groups > 5 to avoid flagging
     benign normal-rate IGMP as flood during high-traffic benign windows.
  5. igmp_join_rate now reflects true join rate (both v2 and v3 destinations).
  6. Added is_239_group() helper to correctly identify group-addressed joins.
  7. Added rounding to all float features for cleaner CSV output.
  8. Added per-PCAP label distribution printout for sanity checking.
  9. Removed attacker_pkts/attacker_ratio — topology-specific, hurts generalization.

Install:
  pip install nfstream pandas numpy
  pip install nfstream pandas numpy --break-system-packages   (on Ubuntu/Debian)

Usage:
  python3 pcap_to_dataset.py

Expected PCAP filenames:
  class0_benign_final.pcap
  class1_igmpflood_final.pcap
  class2_igmpspoof_final.pcap
  class3_mldflood_final.pcap
  class4_pimhello_final.pcap
  class5_mcastamp_final.pcap
  class6_groupscan_final.pcap
  class7_queryflood_final.pcap
  class8_dataflood_final.pcap
"""


'\nNFStream PCAP → Windowed Multicast IDS Training Dataset\n========================================================\nUse Script 2 (Document 16) as the base — it has more features.\nAll corrections applied. See CORRECTIONS section below.\n\nCORRECTIONS vs previous versions:\n  1. igmp_join detection: now captures BOTH IGMPv2 (dst=group addr 239.x.x.x)\n     AND IGMPv3 (dst=224.0.0.22). Previous version missed all benign IGMPv2 joins.\n  2. label_window class 2: added unique_src_ips_igmp > 1 — spoof sends from\n     victim IPs so multiple IGMP sources in one window is the real indicator.\n  3. label_window class 6: added igmp_query_rate check alongside group_churn_rate.\n  4. label_window class 1: added unique_mcast_groups > 5 to avoid flagging\n     benign normal-rate IGMP as flood during high-traffic benign windows.\n  5. igmp_join_rate now reflects true join rate (both v2 and v3 destinations).\n  6. Added is_239_group() helper to correctly identify group-addressed joins.\n  7. Added 

In [9]:
import sys
!{sys.executable} -m pip install nfstream pandas numpy

from nfstream import NFStreamer
import pandas as pd
import numpy as np
import math
import os
import ipaddress
from collections import defaultdict


# =============================================================================
# CONFIG — change these two paths
# =============================================================================

PCAP_DIR   = './pcaps'                         # folder containing your .pcap files
OUTPUT_CSV = './multicast_training_dataset.csv'

WINDOW_SEC = 10.0    # 10-second aggregation window

CLASS_MAP = {
    'class0': 0,    # BENIGN
    'class1': 1,    # IGMP_FLOOD
    'class2': 2,    # IGMP_SPOOF
    'class3': 3,    # MLD_FLOOD
    'class4': 4,    # PIM_HELLO
    'class5': 5,    # MCAST_AMP
    'class6': 6,    # GROUP_SCAN
    'class7': 7,    # QUERY_FLOOD
    'class8': 8,    # DATA_FLOOD
}

LABEL_NAMES = {
    0: 'BENIGN',    1: 'IGMP_FLOOD', 2: 'IGMP_SPOOF',
    3: 'MLD_FLOOD', 4: 'PIM_HELLO',  5: 'MCAST_AMP',
    6: 'GROUP_SCAN',7: 'QUERY_FLOOD',8: 'DATA_FLOOD',
}


# =============================================================================
# HELPERS
# =============================================================================

def is_multicast_ip(ip: str) -> bool:
    """True for any IPv4/IPv6 multicast address."""
    try:
        return ipaddress.ip_address(ip).is_multicast
    except Exception:
        return False


def is_239_group(ip: str) -> bool:
    """
    True for 239.x.x.x — the SSM/ASM group address range used in this lab.
    CORRECTION: IGMPv2 Membership Reports (type=0x16) are sent to the GROUP
    ADDRESS ITSELF (e.g., 239.1.1.1), not to 224.0.0.22.
    224.0.0.22 is the IGMPv3-specific all-routers address.
    Both patterns must be counted as IGMP joins.
    """
    try:
        addr = ipaddress.ip_address(ip)
        return addr.is_multicast and ip.startswith('239.')
    except Exception:
        return False


def r(value: float, digits: int = 4) -> float:
    """Round to avoid floating-point noise in CSV."""
    return round(float(value), digits)


# =============================================================================
# FLOW EXTRACTION
# =============================================================================

def extract_flows(pcap_path: str) -> list:
    """Run NFStream on a PCAP. Returns list of flow objects."""
    flows = []
    try:
        streamer = NFStreamer(
            source         = pcap_path,
            idle_timeout   = 5,
            active_timeout = 30,
            n_dissections  = 0,    # disable DPI — we classify by protocol number
        )
        for flow in streamer:
            flows.append(flow)
    except Exception as e:
        print(f'  ERROR reading {pcap_path}: {e}')
    return flows


# =============================================================================
# WINDOWING
# =============================================================================


def flows_to_windows(
    flow_list,
    window_sec=10.0,
    stride_sec=2.0
):
    """
    Sliding-window aggregation.

    Example:
        window = 10s
        stride = 2s

    Windows:
        0-10
        2-12
        4-14
        ...

    A flow is included in a window if:
        flow.first_seen ∈ [window_start, window_end)

    """

    if not flow_list:
        return {}

    # ---------------------------------------------------------------------
    # Convert to milliseconds
    # ---------------------------------------------------------------------

    window_ms = int(window_sec * 1000)
    stride_ms = int(stride_sec * 1000)

    # ---------------------------------------------------------------------
    # Global timeline
    # ---------------------------------------------------------------------

    min_time = min(
        f.bidirectional_first_seen_ms
        for f in flow_list
    )

    max_time = max(
        f.bidirectional_last_seen_ms
        for f in flow_list
    )

    windows = defaultdict(list)

    # ---------------------------------------------------------------------
    # Sliding windows
    # ---------------------------------------------------------------------

    window_id = 0

    current_start = min_time

    while current_start <= max_time:

        current_end = current_start + window_ms

        # -------------------------------------------------------------
        # Add flows whose first_seen falls inside this window
        # -------------------------------------------------------------

        for f in flow_list:

            ts = f.bidirectional_first_seen_ms

            if current_start <= ts < current_end:

                windows[window_id].append(f)

        # -------------------------------------------------------------
        # Next sliding step
        # -------------------------------------------------------------

        current_start += stride_ms

        window_id += 1

    return windows


# =============================================================================
# FEATURE EXTRACTION
# =============================================================================

def aggregate_window(flow_list: list) -> dict:
    """
    Aggregate all flows in one time window into a single feature vector.
    Returns a flat dict of numeric features.
    """
    if not flow_list:
        return None

    # ── Window timing ─────────────────────────────────────────────────────────
    first_ms = min(f.bidirectional_first_seen_ms for f in flow_list)
    last_ms  = max(f.bidirectional_last_seen_ms  for f in flow_list)
    duration = max((last_ms - first_ms) / 1000.0, 1e-6)

    total_flows = len(flow_list)
    total_pkts  = sum(f.bidirectional_packets for f in flow_list)
    total_bytes = sum(f.bidirectional_bytes   for f in flow_list)
    safe        = max(total_pkts, 1)

    # ── Protocol buckets ──────────────────────────────────────────────────────
    igmp_flows = [f for f in flow_list if f.protocol == 2]
    pim_flows  = [f for f in flow_list if f.protocol == 103]
    ospf_flows = [f for f in flow_list if f.protocol == 89]
    mld_flows  = [f for f in flow_list if f.protocol == 58]
    udp_flows  = [f for f in flow_list if f.protocol == 17]
    tcp_flows  = [f for f in flow_list if f.protocol == 6]
    icmp_flows = [f for f in flow_list if f.protocol == 1]

    igmp_pkts = sum(f.bidirectional_packets for f in igmp_flows)
    pim_pkts  = sum(f.bidirectional_packets for f in pim_flows)
    ospf_pkts = sum(f.bidirectional_packets for f in ospf_flows)
    mld_pkts  = sum(f.bidirectional_packets for f in mld_flows)
    udp_pkts  = sum(f.bidirectional_packets for f in udp_flows)
    tcp_pkts  = sum(f.bidirectional_packets for f in tcp_flows)
    icmp_pkts = sum(f.bidirectional_packets for f in icmp_flows)

    # ── IGMP subtype breakdown ────────────────────────────────────────────────
    # CORRECTION: IGMPv2 join → dst = group address (239.x.x.x)
    #             IGMPv3 join → dst = 224.0.0.22
    #             Both count as joins.
    igmp_join_pkts = sum(
        f.bidirectional_packets for f in igmp_flows
        if str(f.dst_ip) == '224.0.0.22' or is_239_group(str(f.dst_ip or ''))
    )
    igmp_leave_pkts = sum(
        f.bidirectional_packets for f in igmp_flows
        if str(f.dst_ip) == '224.0.0.2'
    )
    igmp_query_pkts = sum(
        f.bidirectional_packets for f in igmp_flows
        if str(f.dst_ip) == '224.0.0.1'
    )

    # ── PIM features ──────────────────────────────────────────────────────────
    # PIM Hellos go to 224.0.0.13 (all PIM routers)
    pim_hello_pkts = sum(
        f.bidirectional_packets for f in pim_flows
        if str(f.dst_ip) == '224.0.0.13'
    )

    # ── MLD features ──────────────────────────────────────────────────────────
    mld_report_pkts = sum(
        f.bidirectional_packets for f in mld_flows
        if str(f.dst_ip or '').startswith('ff02::')
    )

    # ── Multicast group diversity ─────────────────────────────────────────────
    # KEY FEATURE: floods produce hundreds of unique groups; benign uses 2-5
    mcast_group_set = set()
    group_pkt_counter = defaultdict(int)
    for f in flow_list:
        dst = str(f.dst_ip or '')
        if is_multicast_ip(dst):
            mcast_group_set.add(dst)
            group_pkt_counter[dst] += f.bidirectional_packets

    unique_mcast_groups = len(mcast_group_set)

    # ── Group entropy (RFC 1112 semantics) ───────────────────────────────────
    # High entropy = many equally-used groups → IGMP flood / group scan
    # Low entropy  = traffic concentrated on a few groups → benign
    entropy = 0.0
    total_group_pkts = sum(group_pkt_counter.values())
    if total_group_pkts > 0:
        for c in group_pkt_counter.values():
            p = c / total_group_pkts
            if p > 0:
                entropy -= p * math.log2(p)

    # ── Group churn rate ──────────────────────────────────────────────────────
    # Fraction of multicast groups that appear only ONCE in this window.
    # Close to 1.0 for group scan / IGMP flood (each packet a new group).
    # Close to 0.0 for benign (same groups used repeatedly).
    single_use_groups = sum(1 for c in group_pkt_counter.values() if c == 1)
    group_churn_rate  = single_use_groups / max(unique_mcast_groups, 1)

    # ── Group reuse ratio ────────────────────────────────────────────────────
    reused_groups    = sum(1 for c in group_pkt_counter.values() if c > 1)
    group_reuse_ratio = reused_groups / max(unique_mcast_groups, 1)

    # ── Source IP diversity ───────────────────────────────────────────────────
    all_src_ips   = [str(f.src_ip) for f in flow_list]
    igmp_src_ips  = [str(f.src_ip) for f in igmp_flows]

    unique_src_ips      = len(set(all_src_ips))
    unique_src_ips_igmp = len(set(igmp_src_ips))

    # ── Source entropy ────────────────────────────────────────────────────────
    src_counter = defaultdict(int)
    for s in all_src_ips:
        src_counter[s] += 1
    src_entropy = 0.0
    for c in src_counter.values():
        p = c / max(len(all_src_ips), 1)
        if p > 0:
            src_entropy -= p * math.log2(p)

    # ── Multicast UDP (Class 8: Data Flood) ───────────────────────────────────
    mcast_udp_pkts  = sum(f.bidirectional_packets for f in udp_flows
                          if is_multicast_ip(str(f.dst_ip or '')))
    mcast_udp_bytes = sum(f.bidirectional_bytes   for f in udp_flows
                          if is_multicast_ip(str(f.dst_ip or '')))

    # ── Multicast ICMP (Class 5: Amplification) ───────────────────────────────
    mcast_icmp_pkts = sum(f.bidirectional_packets for f in icmp_flows
                          if str(f.dst_ip or '') in ('224.0.0.1', '224.0.0.2'))

    # ── Control vs data ratio ────────────────────────────────────────────────
    control_pkts       = igmp_pkts + pim_pkts + ospf_pkts + mld_pkts
    control_data_ratio = control_pkts / max(mcast_udp_pkts, 1)

    # ── State pressure (RFC 7761: PIM state table stress indicator) ──────────
    # unique_mcast_groups × unique_src_ips = potential (S,G) state entries
    state_pressure = unique_mcast_groups * unique_src_ips

    # ── Query / join ratio ────────────────────────────────────────────────────
    # Normal: close to 0 (very few queries vs many joins)
    # Query flood: very high (all packets are queries)
    query_join_ratio = igmp_query_pkts / max(igmp_join_pkts, 1)

    # ── Multicast fanout ─────────────────────────────────────────────────────
    multicast_fanout = unique_mcast_groups / max(unique_src_ips, 1)

    # ── Packet size statistics ────────────────────────────────────────────────
    pkt_sizes = []
    for f in flow_list:
        if f.bidirectional_packets > 0:
            avg = f.bidirectional_bytes / f.bidirectional_packets
            pkt_sizes.extend([avg] * min(f.bidirectional_packets, 5))

    pkt_size_mean = float(np.mean(pkt_sizes)) if pkt_sizes else 0.0
    pkt_size_std  = float(np.std(pkt_sizes))  if pkt_sizes else 0.0
    pkt_size_min  = float(np.min(pkt_sizes))  if pkt_sizes else 0.0
    pkt_size_max  = float(np.max(pkt_sizes))  if pkt_sizes else 0.0

    # ── Inter-arrival time statistics (flow-level proxy) ─────────────────────
    flow_starts = sorted(f.bidirectional_first_seen_ms / 1000.0 for f in flow_list)
    iats        = [flow_starts[i+1] - flow_starts[i]
                   for i in range(len(flow_starts)-1)]

    iat_mean = float(np.mean(iats)) if iats else 0.0
    iat_std  = float(np.std(iats))  if iats else 0.0
    iat_min  = float(np.min(iats))  if iats else 0.0
    iat_max  = float(np.max(iats))  if iats else 0.0

    # Burstiness = coefficient of variation of IAT
    # High for burst-pause attacks, low for steady benign traffic
    burstiness = iat_std / max(iat_mean, 1e-9)

    # ── Assemble feature vector ───────────────────────────────────────────────
    return {
        # Window metadata
        'window_duration_s':    r(duration, 4),

        # Volume
        'total_flows':          total_flows,
        'total_pkts':           total_pkts,
        'total_bytes':          total_bytes,
        'pkt_rate':             r(total_pkts  / duration),
        'byte_rate':            r(total_bytes / duration),
        'flow_rate':            r(total_flows / duration),

        # Per-protocol packet counts
        'igmp_pkts':            igmp_pkts,
        'pim_pkts':             pim_pkts,
        'ospf_pkts':            ospf_pkts,
        'mld_pkts':             mld_pkts,
        'udp_pkts':             udp_pkts,
        'tcp_pkts':             tcp_pkts,
        'icmp_pkts':            icmp_pkts,

        # Per-protocol rates (pkts/s)
        'igmp_rate':            r(igmp_pkts  / duration),
        'pim_rate':             r(pim_pkts   / duration),
        'ospf_rate':            r(ospf_pkts  / duration),
        'mld_rate':             r(mld_pkts   / duration),

        # Per-protocol ratios (fraction of total packets)
        'igmp_ratio':           r(igmp_pkts  / safe),
        'pim_ratio':            r(pim_pkts   / safe),
        'udp_ratio':            r(udp_pkts   / safe),
        'tcp_ratio':            r(tcp_pkts   / safe),
        'icmp_ratio':           r(icmp_pkts  / safe),

        # IGMP subtype features
        'igmp_join_pkts':       igmp_join_pkts,
        'igmp_leave_pkts':      igmp_leave_pkts,
        'igmp_query_pkts':      igmp_query_pkts,
        'igmp_join_rate':       r(igmp_join_pkts  / duration),
        'igmp_leave_rate':      r(igmp_leave_pkts / duration),
        'igmp_query_rate':      r(igmp_query_pkts / duration),

        # PIM features
        'pim_hello_pkts':       pim_hello_pkts,
        'pim_hello_rate':       r(pim_hello_pkts  / duration),

        # MLD features
        'mld_report_pkts':      mld_report_pkts,
        'mld_report_rate':      r(mld_report_pkts / duration),

        # Multicast group features (KEY for flood / scan detection)
        'unique_mcast_groups':  unique_mcast_groups,
        'mcast_group_rate':     r(unique_mcast_groups / duration),
        'group_entropy':        r(entropy),
        'group_churn_rate':     r(group_churn_rate),
        'group_reuse_ratio':    r(group_reuse_ratio),

        # Source IP diversity features (KEY for spoofing detection)
        'unique_src_ips':       unique_src_ips,
        'unique_src_ips_igmp':  unique_src_ips_igmp,
        'src_ip_entropy':       r(src_entropy),

        # Derived ratio features
        'control_data_ratio':   r(control_data_ratio),
        'state_pressure':       state_pressure,
        'query_join_ratio':     r(query_join_ratio),
        'multicast_fanout':     r(multicast_fanout),

        # Multicast UDP features (Class 8: Data Flood)
        'mcast_udp_pkts':       mcast_udp_pkts,
        'mcast_udp_bytes':      mcast_udp_bytes,
        'mcast_udp_rate':       r(mcast_udp_pkts / duration),

        # Multicast ICMP features (Class 5: Amplification)
        'mcast_icmp_pkts':      mcast_icmp_pkts,
        'mcast_icmp_rate':      r(mcast_icmp_pkts / duration),

        # Packet size statistics
        'pkt_size_mean':        r(pkt_size_mean),
        'pkt_size_std':         r(pkt_size_std),
        'pkt_size_min':         r(pkt_size_min),
        'pkt_size_max':         r(pkt_size_max),

        # Inter-arrival time statistics
        'iat_mean_s':           r(iat_mean, 6),
        'iat_std_s':            r(iat_std,  6),
        'iat_min_s':            r(iat_min,  6),
        'iat_max_s':            r(iat_max,  6),
        'burstiness':           r(burstiness),
    }


# =============================================================================
# PER-WINDOW LABELING
# =============================================================================

def label_window(features: dict, pcap_class: int) -> int:
    """
    Per-window labeling with corrected thresholds.

    Design principle (from CICIDS 2017 methodology):
      Even within an attack-class PCAP, windows that fall in pause phases
      or contain only background traffic get labeled BENIGN (0).
      Only windows that show the actual attack signature get the attack label.

    This prevents the model from learning "capture context" instead of
    actual attack behavior.

    Threshold rationale:
      - Benign igmp_rate: 0-2 pkt/s (5 groups × refresh every 30-120s)
      - Attack igmp_rate: 100-5000 pkt/s
      - Threshold 20 pkt/s = well above benign ceiling, below attack floor
    """

    if pcap_class == 0:
        return 0   # All windows in benign PCAP are benign

    if pcap_class == 1:   # IGMP Flood
        # CORRECTION: require BOTH high rate AND many groups.
        # High igmp_rate alone could appear in benign if router sends query
        # and many hosts respond simultaneously.
        # Many unique_mcast_groups (>5) is the definitive flood indicator —
        # benign uses at most 5 fixed groups.
        if (features['igmp_rate'] > 20 and
                features['unique_mcast_groups'] > 5):
            return 1
        return 0

    if pcap_class == 2:   # IGMP Spoof
        # CORRECTION: require unique_src_ips_igmp > 1.
        # The spoof attack forges Leave packets from victim IPs (192.168.1.20,
        # 192.168.2.10, 192.168.2.20) — different from attacker (192.168.1.10).
        # So multiple IGMP source IPs in one window = spoofing indicator.
        # Leave rate > 5 alone is not enough — benign hosts occasionally leave.
        if (features['igmp_leave_rate'] > 5 and
                features['unique_src_ips_igmp'] > 1):
            return 2
        return 0

    if pcap_class == 3:   # MLD Flood
        if features['mld_rate'] > 20:
            return 3
        return 0

    if pcap_class == 4:   # PIM Hello Manipulation
        # Multiple unique source IPs sending PIM = fake neighbors
        if (features['pim_rate'] > 5 and
                features['unique_src_ips'] > 3):
            return 4
        return 0

    if pcap_class == 5:   # Multicast Amplification
        # ICMP to all-hosts multicast at attack rate
        if features['mcast_icmp_rate'] > 5:
            return 5
        return 0

    if pcap_class == 6:   # Group Scanning
        # CORRECTION: require BOTH group_churn_rate AND igmp_query_rate.
        # group_churn_rate alone is high for IGMP flood too.
        # igmp_query_rate > 5 is specific to group scan (queries from a host,
        # not a router, at flood rate).
        if (features['group_churn_rate'] > 0.8 and
            features['igmp_query_rate'] > 20 and
            features['unique_mcast_groups'] > 20 and
            features['group_entropy'] > 3):
            return 6
        return 0

    if pcap_class == 7:   # IGMP Query Flood
        # General queries (gaddr=0.0.0.0) at high rate
        if features['igmp_query_rate'] > 100:
            return 7
        return 0

    if pcap_class == 8:   # Multicast Data Flood
        # High-rate UDP directly to multicast group from non-server
        if features['mcast_udp_rate'] > 20 :
            return 8

    return 0


# =============================================================================
# MAIN
# =============================================================================

def multicast_flows(pcap_file,pcap_class):
    all_rows = []
    pcap_filename = os.path.splitext(os.path.basename(pcap_file))[0]
    flows = extract_flows(pcap_file)
    if not flows:
        print(f'  WARNING: zero flows extracted — skipping')
        return
    # ... (previous code)
    flow_OUTPUT_CSV = f"{pcap_filename}_class{pcap_class}_nsf_flow.csv"

    # Convert the list to a Pandas DataFrame
    df_flows = NFStreamer(source=pcap_file).to_pandas()

    # Now you can use to_csv
    df_flows.to_csv(flow_OUTPUT_CSV, index=False)
    # ... (rest of code)

    print(f'  NFStream flows:  {len(flows)}')
    windows = flows_to_windows(flows, window_sec=10.0, stride_sec=2.0)
    print(f'  Time windows:    {len(windows)}  ({WINDOW_SEC}s each)')
    # Protocol composition sanity check
    protocols = defaultdict(int)
    for f in flows:
        protocols[f.protocol] += f.bidirectional_packets
    proto_names = {2:'IGMP', 6:'TCP', 17:'UDP', 58:'ICMPv6/MLD',
                   89:'OSPF', 103:'PIM', 1:'ICMP'}
    top_protos = sorted(protocols.items(), key=lambda x: -x[1])[:6]
    print('  Protocol mix:   ', end='')
    print('  '.join(f'{proto_names.get(p,p)}={c}' for p,c in top_protos))
    window_rows = []
    for bucket_id, window_flows in sorted(windows.items()):
        features = aggregate_window(window_flows)
        if features is None:
            continue
        label      = label_window(features, pcap_class)
        label_name = LABEL_NAMES[label]
        row = {
            'source_pcap': pcap_file,
            'pcap_class':  pcap_class,
            'bucket_id':   bucket_id,
            'label':       label,
            'label_name':  label_name,
            **features
        }
        window_rows.append(row)
    # Per-PCAP label distribution
    if window_rows:
        df_check = pd.DataFrame(window_rows)
        dist     = df_check['label_name'].value_counts()
        print(f'\n  Window label distribution:')
        for lname, cnt in dist.items():
            pct = cnt / len(df_check) * 100
            bar = '█' * int(pct / 3)
            print(f'    {lname:<22} {cnt:>5} windows  ({pct:.1f}%)  {bar}')
        # Warn if expected attack class has zero windows
        expected_attack = (df_check['label'] == pcap_class).sum()
        if pcap_class != 0 and expected_attack == 0:
            print(f'\n  ⚠ WARNING: zero windows labeled as class {pcap_class}')
            print(f'    Likely cause: attack rate too low or threshold too high')
            print(f'    Check aggregate features for this PCAP:')
            print(f'    igmp_rate max  = {df_check["igmp_rate"].max():.2f}')
            print(f'    mld_rate max   = {df_check["mld_rate"].max():.2f}')
            print(f'    pim_rate max   = {df_check["pim_rate"].max():.2f}')
            print(f'    mcast_udp_rate = {df_check["mcast_udp_rate"].max():.2f}')
            print(f'    igmp_query_rate= {df_check["igmp_query_rate"].max():.2f}')
            print(f'    unique_mcast_groups max = {df_check["unique_mcast_groups"].max()}')
        else:
            print(f'\n  ✓ {expected_attack} windows confirmed as class {pcap_class}')
    all_rows.extend(window_rows)

    if not all_rows:
        print('\nNo rows collected — check PCAP directory and filenames.')
        return

    # ── Build master dataset ─────────────────────────────────────────────────
    master = pd.DataFrame(all_rows)

    meta_cols    = ['source_pcap', 'pcap_class', 'bucket_id', 'label_name']
    feature_cols = [c for c in master.columns if c not in meta_cols + ['label']]

    print(f'\n{"="*65}')
    print(f'MASTER DATASET SUMMARY')
    print(f'{"="*65}')
    print(f'  Total windows:  {len(master)}')
    print(f'  Features:       {len(feature_cols)}')

    print(f'\n  Label distribution:')
    dist = master['label_name'].value_counts()
    for lname, cnt in dist.items():
        pct = cnt / len(master) * 100
        bar = '█' * int(pct / 2)
        print(f'    {lname:<22} {cnt:>6} ({pct:5.1f}%)  {bar}')

    benign_n = (master['label'] == 0).sum()
    attack_n = (master['label'] != 0).sum()
    ratio    = benign_n / max(attack_n, 1)
    print(f'\n  Benign: {benign_n}   Attack: {attack_n}   Ratio: {ratio:.2f}:1')

    if ratio < 0.3:
        print('  ⚠ Very few benign windows. Run more benign captures.')
    elif ratio > 10:
        print('  ⚠ Very few attack windows. Check labeling thresholds.')
    else:
        print('  ✓ Class balance acceptable for training.')

    # Feature stats (for verifying benign vs attack separation)
    print(f'\n  Key feature ranges (benign vs attack):')
    key_features = ['igmp_rate', 'pim_rate', 'mld_rate', 'unique_mcast_groups',
                    'group_entropy', 'group_churn_rate', 'mcast_udp_rate',
                    'mcast_icmp_rate', 'igmp_query_rate', 'unique_src_ips_igmp']
    benign_df = master[master['label'] == 0]
    attack_df = master[master['label'] != 0]
    print(f'  {"Feature":<28} {"Benign max":>12} {"Attack mean":>12}')
    print(f'  {"-"*52}')
    for feat in key_features:
        if feat in master.columns:
            b_max = benign_df[feat].max() if len(benign_df) > 0 else 0
            a_mean = attack_df[feat].mean() if len(attack_df) > 0 else 0
            print(f'  {feat:<28} {b_max:>12.3f} {a_mean:>12.3f}')

    master_OUTPUT_CSV = f"{pcap_filename}_class{pcap_class}_window_flow.csv"
    # Convert the 'master' list into a Pandas DataFrame
    master_df = pd.DataFrame(master)
    # Save the DataFrame to CSV
    master_df.to_csv(master_OUTPUT_CSV, index=False)
    print(f'\n  Saved → {master_OUTPUT_CSV}')
    print(f'  Feature columns: {feature_cols}')



In [12]:
multicast_flows("/content/class8_r1-sw1.pcap",8)

  NFStream flows:  3200
  Time windows:    291  (10.0s each)
  Protocol mix:   UDP=57258  TCP=3489  ICMP=1081  OSPF=53  PIM=17  IGMP=12

  Window label distribution:
    BENIGN                   165 windows  (56.7%)  ██████████████████
    DATA_FLOOD               126 windows  (43.3%)  ██████████████

  ✓ 126 windows confirmed as class 8

MASTER DATASET SUMMARY
  Total windows:  291
  Features:       59

  Label distribution:
    BENIGN                    165 ( 56.7%)  ████████████████████████████
    DATA_FLOOD                126 ( 43.3%)  █████████████████████

  Benign: 165   Attack: 126   Ratio: 1.31:1
  ✓ Class balance acceptable for training.

  Key feature ranges (benign vs attack):
  Feature                        Benign max  Attack mean
  ----------------------------------------------------
  igmp_rate                           0.229        0.007
  pim_rate                            0.131        0.011
  mld_rate                            0.105        0.001
  unique_mcast_gro